# Publicando os passes por Copa

## Task 4: publicar a tabela `copa.passes_por_copa` e conferir o job na Spark UI

## O problema

Gravar `passes_por_copa` como tabela e conferir na Spark UI o que o Spark fez para gerá-la.

In [ ]:
import sys
sys.path.insert(0, "../../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("arena-tabela-passes")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "../../data/warehouse")
    .getOrCreate()
)

RAW = "../../data/raw"
EVENTOS = "../../data/events"

spark

A célula abaixo já vem pronta: o código dos vídeos 4.1 a 4.3 (lances, partidas, ano e passes por partida e por Copa).

In [ ]:
# vídeo 4.1: os lances e as partidas
from wc_pipeline.schemas import MATCHES
from wc_pipeline.transformacoes import limpar_partidas

eventos = spark.read.parquet(EVENTOS).withColumnRenamed("MatchID", "match_id")

partidas = limpar_partidas(
    spark.read.option("header", "true").schema(MATCHES).csv(f"{RAW}/WorldCupMatches.csv")
)

# vídeo 4.2: o ano em cada lance
lances = eventos.join(partidas.select("match_id", "ano"), on="match_id")

# vídeo 4.3: os passes por partida e por Copa
passes_por_partida = (
    lances
    .filter(F.col("event_type") == "Pass")
    .groupBy("ano", "match_id", "team")
    .agg(F.count("*").alias("passes"))
)

passes_por_copa = (
    passes_por_partida
    .groupBy("ano", F.col("team").alias("selecao"))
    .agg(
        F.sum("passes").alias("passes"),
        F.count("*").alias("partidas"),
        F.round(F.avg("passes"), 1).alias("media_por_partida"),
    )
)

A próxima célula também vem pronta: ela só apaga a pasta da tabela desta task, para o notebook poder rodar de novo
(vídeo 3.4).

In [ ]:
import shutil
shutil.rmtree("../../data/warehouse/copa.db/passes_por_copa", ignore_errors=True)

## Publicando a tabela

## O que o job fez: a Spark UI

Spark UI (http://localhost:4040), aba **SQL / DataFrame**, consulta com a descrição `saveAsTable`. Anote o número de
cada caixa:

| Caixa | Métrica | O que ela mostra |
|---|---|---|
| `Scan parquet` | number of output rows | quantos lances a leitura trouxe |
| `Filter` | number of output rows | quantos passes sobraram depois do filtro |
| `BroadcastExchange` | number of output rows | quantas partidas foram copiadas para os executors |
| `Exchange` (o primeiro) | shuffle records written | quantas linhas atravessaram o shuffle |
| `AQEShuffleRead` | number of partitions | em quantas partições o AQE juntou as 200 |
| `Execute InsertIntoHadoopFsRelationCommand` | number of written files | quantos arquivos a tabela ganhou |

## Demanda da aula 4 atendida

| Etapa | Entrega |
|---|---|
| Task 1 | os gols do feed conferidos com os placares |
| Task 2 | `lances`, com o ano da Copa em cada lance |
| Task 3 | `passes_por_copa`, com passes, partidas e média |
| Task 4 | `copa.passes_por_copa` publicada e o job conferido na Spark UI |

## Para consultar depois

- **Cache:** `df.cache()` guarda o resultado de um DataFrame na memória da primeira vez que uma ação o calcula; as
  ações seguintes leem dali. Vale quando um resultado **caro** é usado **várias vezes**, como os lances com o ano usados
  em várias estatísticas. Para um resultado pequeno, pode até atrasar. Confira na aba **Storage** da Spark UI e libere
  com `df.unpersist()`.
- **Aba Stages:** mostra cada stage com o tempo de cada task (**Min**, **Median**, **Max**). Uma task muito mais lenta
  que as outras costuma ser **data skew**: uma chave com muito mais linhas que as outras, e o shuffle manda todas para
  a mesma task.
- **`spark.table("copa.passes_por_copa")`** lê a tabela do catálogo como DataFrame.

## Encerrando

Ao terminar o notebook, encerre a sessão Spark. Isso libera a memória e a porta da Spark UI para o próximo notebook.

In [ ]:
spark.stop()